# Create focus-test Dave config

Builds the focus-lock test Dave recipe: visits every FOV and checks the
focus lock, with no fluidics, before the full acquisition. Needs only the
positions (02b) and notebook 01's focus-test HAL config. Run its recipe on
the microscope first; then (tumor_disk) `02d_filter_positions_by_tissue_quality.ipynb`,
then 03 and 04 on the kept FOVs.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent  # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave   import (
    dave_focustest_config_filename, create_focus_test_dave_config, experiment_estimate_table,
)
from MERci.acquisition.kilroy import find_kilroy_config
from MERci.acquisition.display import display_xml
from MERci.analysis.stage_z   import read_off_file_if_ready, summarize_focus_lock
from MERci.acquisition.pipeline_config import load_pipeline_config

PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"

# Positions files are named with POSITIONS_TAG, the Dave recipe with SAMPLE_NAME.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE    = PIPELINE_CONFIG.microscope
# Focus-test movie length: 0 = check focus only, no movie. tumor_disk takes a
# 405 + blank movie per FOV, the input of 02d_filter_positions_by_tissue_quality.
N_TEST_FRAMES = PIPELINE_CONFIG.focus_test_frames

# Resolve the Kilroy config that will run this experiment (same as notebook 04).
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"MICROSCOPE    : {MICROSCOPE}")
print(f"N_TEST_FRAMES : {N_TEST_FRAMES}")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

## Focus-lock test recipe

A separate Dave recipe, run before notebook 04's recipes: visits every FOV
and checks the focus lock, with no fluidics, to catch a bad lock before the
full acquisition.

`FOCUS_TEST_LEAD_MIN` is when the recipe is expected to start, for the timing
estimate.

`N_TEST_FRAMES` (pipeline.yaml's `focus_test_frames`):
- `0` -- check focus only, no image. Nothing is saved per FOV: watch Dave's
  warnings panel while the recipe runs.
- `> 0` -- also takes a short movie per FOV, with notebook 01's focus-test HAL
  config. Each movie's `.off` file records the lock per frame
  (`good-offset`); the last cell reads them back. 02d scores the same movies.

**Positions:** `positions_{POSITIONS_TAG}_all.txt` if 02d has run (it keeps
the full list there), else `positions_{POSITIONS_TAG}.txt`.

In [ ]:
# ── Focus-lock test parameters ──────────────────────────────────────────
FOCUS_TEST_LEAD_MIN = 30    # minutes until the recipe starts (timing estimate only)
NUM_FOCUS_CHECKS = 50
FOCUS_SCAN       = True

# Focus-test HAL config from notebook 01.
_focustest_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*-focustest-*.xml"))
if not _focustest_hal_configs:
    raise FileNotFoundError(
        f"No focus-test HAL config found in {SETTINGS_DIR} -- run notebook 01's "
        f"'create hal config for focus test' section first."
    )
TEST_HAL_CONFIG = _focustest_hal_configs[-1].name
print(f"Focus-test HAL config: {TEST_HAL_CONFIG}")

# After 02d, the full FOV list is the _all file.
focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_all.txt"
if not focus_test_positions.exists():
    focus_test_positions = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
if not focus_test_positions.exists():
    raise FileNotFoundError(f"{focus_test_positions} not found -- run 02b first.")

focus_test_output   = SETTINGS_DIR / dave_focustest_config_filename(MICROSCOPE, SAMPLE_NAME)
focus_test_data_dir = (SAMPLE_DIR / "data" / "focus_test") if N_TEST_FRAMES > 0 else None

focus_test_start = datetime.now() + timedelta(minutes=FOCUS_TEST_LEAD_MIN)

n_fovs, focustest_est = create_focus_test_dave_config(
    positions_file   = focus_test_positions,
    output_path      = focus_test_output,
    num_focus_checks = NUM_FOCUS_CHECKS,
    focus_scan       = FOCUS_SCAN,
    n_test_frames    = N_TEST_FRAMES,
    hal_config       = TEST_HAL_CONFIG,
    settings_dir     = SETTINGS_DIR,
    data_dir         = focus_test_data_dir,
    movie_name       = f"hal-{MICROSCOPE.lower()}-focustest",
    kilroy_config    = KILROY_CONFIG,
    microscope       = MICROSCOPE,
    start_time       = focus_test_start,
    per_round        = False,   # per-round breakdown shown as a table at the end instead
)

print(f"Focus-lock test recipe saved: {focus_test_output}")
print(f"Positions: {focus_test_positions.name}")
print(f"FOVs visited: {n_fovs}")
if N_TEST_FRAMES > 0:
    print(f"Each FOV takes a real {N_TEST_FRAMES}-frame movie -> .off files land under {focus_test_data_dir}")
else:
    print("Check-focus only -- no movie taken, no persisted per-FOV record (see markdown above).")

display_xml(focus_test_output)

In [ ]:
display(experiment_estimate_table(focustest_est, focus_test_start))

In [ ]:
# ── Read back per-FOV focus-lock results ──────────────────────────────────
# Needs N_TEST_FRAMES > 0 and the recipe run on the microscope.
if N_TEST_FRAMES > 0:
    # Dave zero-pads the FOV index to the digits of the FOV count.
    pad = len(str(n_fovs))
    bad_fovs  = []
    n_checked = 0
    for fov_idx in range(n_fovs):
        off_path = focus_test_data_dir / f"hal-{MICROSCOPE.lower()}-focustest_{str(fov_idx).zfill(pad)}.off"
        # None = not written yet, or still being written.
        off_df = read_off_file_if_ready(off_path)
        if off_df is None:
            continue
        n_checked += 1
        summary = summarize_focus_lock(off_df)
        if not summary["all_good"]:
            bad_fovs.append((fov_idx, summary))

    print(f"Checked {n_checked} / {n_fovs} FOV(s) (missing ones haven't been imaged yet).")
    if bad_fovs:
        print(f"{len(bad_fovs)} FOV(s) with a bad focus lock:")
        for fov_idx, summary in bad_fovs:
            print(f"  FOV {fov_idx}: {summary['n_bad_frames']}/{summary['n_frames']} bad frame(s)")
    elif n_checked:
        print("All checked FOVs had a good focus lock.")
    else:
        print("No FOVs found yet -- run the recipe on the microscope first.")
else:
    print("N_TEST_FRAMES == 0 -- no per-FOV file was written (see markdown above); "
          "check Dave's own warnings panel while the recipe runs instead.")